In [ ]:
import argparse
import csv
import html
import re
from pathlib import Path

KEYWORDS = {
    'excellent': 'Expected positive',
    'great': 'Expected positive',
    'wonderful': 'Expected positive',
    'awful': 'Expected negative',
    'terrible': 'Expected negative',
    'boring': 'Expected negative',
}


def tokenize(review):
    text = re.sub(r'<[^>]+>', ' ', html.unescape(review))
    return set(re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower()))


def count_reviews(csv_path):
    total = 0
    positive = 0

    counts = {
        word: {"all": 0, "positive": 0}
        for word in KEYWORDS
    }

    with open(csv_path, encoding="utf-8-sig", newline="") as source:
        reader = csv.DictReader(source)

        if not {"review", "sentiment"} <= set(reader.fieldnames or []):
            raise ValueError(
                "CSV must contain review and sentiment columns."
            )

        for line, row in enumerate(reader, start=2):
            raw_label = row.get("sentiment")

            if raw_label is None or not raw_label.strip():
                raise ValueError(
                    f"Missing sentiment in CSV record {line}. "
                    "Check the file or download the dataset again."
                )

            label = raw_label.strip().lower()

            if label not in {"positive", "negative"}:
                raise ValueError(
                    f"Unexpected sentiment in CSV record {line}: {label!r}"
                )

            review = row.get("review")

            if review is None or not review.strip():
                raise ValueError(
                    f"Missing review text in CSV record {line}."
                )

            total += 1
            positive += label == "positive"

            words = tokenize(review)

            for word in KEYWORDS:
                if word in words:
                    counts[word]["all"] += 1
                    counts[word]["positive"] += label == "positive"

    if total == 0 or positive == 0:
        raise ValueError(
            "Need a nonempty dataset with positive reviews."
        )

    return total, positive, counts


def calculate_probabilities(total, positive, counts):
    results = []
    for word, count in counts.items():
        prior = positive / total
        likelihood = count['positive'] / positive
        marginal = count['all'] / total
        # Bayes: P(Positive | word) = P(word | Positive) P(Positive) / P(word)
        posterior = likelihood * prior / marginal if marginal else None
        results.append({
            'keyword': word, 'selection': KEYWORDS[word],
            'reviews_with_keyword': count['all'],
            'positive_reviews_with_keyword': count['positive'],
            'prior': prior, 'likelihood': likelihood,
            'marginal': marginal, 'posterior': posterior,
        })
    return results


def markdown_table(results):
    lines = [
        '| Keyword | Reviews containing word | Positive containing word | P(Positive) | P(word given Positive) | P(word) | P(Positive given word) |',
        '|---|---:|---:|---:|---:|---:|---:|',
    ]
    for row in results:
        probabilities = [f"{row[key]:.6f}" if row[key] is not None else 'undefined'
                         for key in ('prior', 'likelihood', 'marginal', 'posterior')]
        lines.append('| ' + ' | '.join([
            row['keyword'], str(row['reviews_with_keyword']),
            str(row['positive_reviews_with_keyword']), *probabilities]) + ' |')
    return '\n'.join(lines)



In [6]:
DATASET = Path("IMDB Dataset.csv")
total, positive, counts = count_reviews(DATASET)
results = calculate_probabilities(total, positive, counts)
print(f"Reviews: {total:,}; positive: {positive:,}; negative: {total-positive:,}")
print(markdown_table(results))


Reviews: 50,000; positive: 25,000; negative: 25,000
| Keyword | Reviews containing word | Positive containing word | P(Positive) | P(word given Positive) | P(word) | P(Positive given word) |
|---|---:|---:|---:|---:|---:|---:|
| excellent | 3552 | 2868 | 0.500000 | 0.114720 | 0.071040 | 0.807432 |
| great | 12576 | 8477 | 0.500000 | 0.339080 | 0.251520 | 0.674062 |
| wonderful | 2778 | 2256 | 0.500000 | 0.090240 | 0.055560 | 0.812095 |
| awful | 2882 | 283 | 0.500000 | 0.011320 | 0.057640 | 0.098196 |
| terrible | 2699 | 382 | 0.500000 | 0.015280 | 0.053980 | 0.141534 |
| boring | 3052 | 592 | 0.500000 | 0.023680 | 0.061040 | 0.193971 |
